In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))



Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12
Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


In [5]:


import plotly.graph_objects as go

asia_df = df[df['Region'] == 'Asia']
asian_countries = asia_df['Country'].unique()

highlight_country = 'China'

fig = go.Figure()

for country in asian_countries:
    country_data = asia_df[asia_df['Country'] == country]

    if country == highlight_country:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            line=dict(color='#1D3557', width=4),
            name=country,
            showlegend=False
        ))

        last_point = country_data.iloc[-1]
        fig.add_annotation(
            x=last_point['Year'],
            y=last_point['CO2_Mt'],
            text=f"  <b>{country}</b>",
            showarrow=False,
            xanchor='left',
            font=dict(color='#1D3557', size=14, family="Arial")
        )
    else:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            line=dict(color='#DDDDDD', width=1.5),
            name=country,
            hoverinfo='name+y',
            showlegend=False
        ))

fig.update_layout(
    title={
        'text': "<b>China's CO2 Emissions Tripled Since 2000, Far Outpacing Other Asian Economies</b>",
        'font': {'size': 20, 'family': 'Arial'},
        'x': 0.05,
        'y': 0.95
    },
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', color='#333333'),
    xaxis=dict(
        title="Year",
        showgrid=False,
        linecolor='#333333',
        dtick=5
    ),
    yaxis=dict(
        title="CO2 Emissions (Mt)",
        showgrid=True,
        gridcolor='#F0F0F0',
        linecolor='white'
    ),
    margin=dict(r=120, t=100, l=60, b=60) # Extra right margin for direct labels
)

fig.show()

In [15]:
import plotly.graph_objects as go


regional_avg = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()
slope_df = regional_avg[regional_avg['Year'].isin([2000, 2022])]

fig = go.Figure()

COLOR_UP = "#E63946"
COLOR_DOWN = "#2A9D8F"

for region in slope_df['Region'].unique():
    region_data = slope_df[slope_df['Region'] == region].sort_values('Year')
    val_2000 = region_data[region_data['Year'] == 2000]['CO2_Mt'].values[0]
    val_2022 = region_data[region_data['Year'] == 2022]['CO2_Mt'].values[0]

    color = COLOR_UP if val_2022 > val_2000 else COLOR_DOWN


    fig.add_trace(go.Scatter(
        x=region_data['Year'].astype(str),
        y=region_data['CO2_Mt'],
        mode='lines+markers+text',
        line=dict(color=color, width=3),
        marker=dict(size=10),
        showlegend=False,
        text=[f'{region_data["CO2_Mt"].iloc[0]:.0f}', f'{region}<br>{region_data["CO2_Mt"].iloc[1]:.0f}'],
        textposition=['middle left', 'middle right'],
        textfont=dict(size=10, color=color, family='Arial')
    ))


    fig.add_annotation(
        x="2000", y=val_2000,
        text=f"{region}  <b>{val_2000:.0f}</b>",
        showarrow=False,
        xanchor='right',
        xshift=-15,
        font=dict(size=12, color=color, family="Arial")
    )


    fig.add_annotation(
        x="2022", y=val_2022,
        text=f"<b>{val_2022:.0f}</b>",
        showarrow=False,
        xanchor='left',
        xshift=15,
        font=dict(size=12, color=color, family="Arial")
    )


fig.update_layout(
    title={
        'text': "<b>Asia's Emissions Skyrocketed Since 2000, While Western Regions Achieved Reductions</b>",
        'font': {'size': 20, 'family': 'Arial'},
        'x': 0.05, 'y': 0.95
    },
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        showgrid=False,
        linecolor='white',
        tickfont=dict(size=14, family="Arial", weight='bold'),
        range=[-0.6, 1.6]
    ),
    yaxis=dict(
        showgrid=False,
        showticklabels=False,
        zeroline=False
    ),

    margin=dict(l=250, r=250, t=100, b=50),
    height=600
)

fig.show()